# Practice Lab: Self-Hosting with Ollama (Lesson 13.5)

Module 13 . 4 exercises . Benchmark + local RAG + hybrid router + Cloud Run deploy


# Lesson 13.5: Self-Hosting with Ollama

4 exercises: 1E + 2M + 1C

Module 13: Cost Optimization & Efficiency


In [ ]:
import random, re
random.seed(42)


---
## Exercise 1 (Easy): Model Benchmark


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

MODELS={"gemma3:1b":{"ram":4,"q":6,"cpu":50,"gpu":150},"gemma3:4b":{"ram":6,"q":7.5,"cpu":30,"gpu":100},
    "llama3.1:8b":{"ram":10,"q":8,"cpu":15,"gpu":80},"qwen3:14b":{"ram":18,"q":8.5,"cpu":5,"gpu":60},
    "llama3.3:70b":{"ram":48,"q":9,"cpu":0,"gpu":30}}
HW={"8GB Laptop":{"ram":8,"gpu":False},"16GB Laptop":{"ram":16,"gpu":False},"L4 GPU":{"ram":32,"gpu":True}}

print("Ollama Benchmark:")
for hw,cfg in HW.items():
    print(f"\n  {hw} (RAM:{cfg['ram']}GB GPU:{cfg['gpu']}):")
    print(f"  {'Model':<16} {'Fits':>5} {'TPS':>5} {'200tok':>7} {'Q':>4}")
    best=None; bq=0
    for m,info in MODELS.items():
        fits=info["ram"]<=cfg["ram"]; tps=info["gpu"] if cfg["gpu"] else info["cpu"]
        if not fits or tps==0: print(f"  {m:<16} {'No':>5} {'--':>5} {'--':>7} {info['q']:>3}"); continue
        lat=round(200/max(tps+random.gauss(0,tps*0.05),1)*1000)
        print(f"  {m:<16} {'Yes':>5} {tps:>5} {lat:>6}ms {info['q']:>3}")
        if info["q"]>bq: bq=info["q"]; best=m
    if best: print(f"  Best: {best} ({bq}/10)")
print(f"\n  API Flash: 2200ms for 200tok, $0.00005/call")


</details>


---
## Exercise 2 (Medium): Local RAG


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class LRAG:
    def __init__(self): self.docs=[]
    def index(self,n,cpd=5):
        tc=n*cpd; et=round(tc/500,2); self.docs=list(range(tc))
        return {"docs":n,"chunks":tc,"time_s":et,"mb":round(tc*768*4/1e6,1)}
    def query(self):
        em=2; rt=max(1,round(2+len(self.docs)*0.01+random.gauss(0,1)))
        gn=max(500,round(200/30*1000+random.gauss(0,100)))
        return {"embed_ms":em,"retrieve_ms":rt,"gen_ms":gn,"total_ms":em+rt+gn,"cost":0}

r=LRAG(); idx=r.index(50)
print("Local RAG Pipeline:")
print(f"  Indexed: {idx['docs']} docs -> {idx['chunks']} chunks in {idx['time_s']}s ({idx['mb']}MB)")
lats=[]
print(f"  {'Q#':>4} {'Embed':>6} {'Ret':>5} {'Gen':>6} {'Total':>7}")
for i in range(20):
    q=r.query(); lats.append(q["total_ms"])
    if i<3 or i>=18: print(f"  {i+1:>4} {q['embed_ms']:>5}ms {q['retrieve_ms']:>4}ms {q['gen_ms']:>5}ms {q['total_ms']:>6}ms")
    elif i==3: print(f"  {'...':>4}")

al=sum(lats)/20; cc=20*(3050*0.10/1e6+200*0.40/1e6)
print(f"\n  Local: avg={al:.0f}ms, cost=$0 | Cloud: ~300ms, ${cc:.6f}")
print(f"  Monthly 1K/day: local=Rs0 | cloud=Rs{cc/20*1000*30*84:.0f}")


</details>


---
## Exercise 3 (Medium): Hybrid Router


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random, re; random.seed(42)

class HR:
    SP=[(r"^(hi|hello|thanks|bye|ok)[!.\s]*$","greeting"),(r"^(what|who|when|where)\s+(is|are)","factual"),
        (r"(how much|price|cost)","pricing"),(r"(refund|cancel)","faq")]
    CP=[(r"(compare|contrast|versus|vs)","analysis"),(r"(design|architect|build).+(system|arch|platform)","complex"),
        (r"(step.by.step|in detail)","reasoning"),(r"(write|generate).+(report|essay|article)","creative"),
        (r"(analyze|review).+(code|performance|bug)","analysis")]
    def __init__(self): self.s={"t":0,"l":0,"a":0,"ac":0}
    def route(self,q):
        self.s["t"]+=1; ql=q.lower().strip()
        for p,c in self.SP:
            if re.search(p,ql,re.I): self.s["l"]+=1; return {"d":"ollama","c":0}
        # Check complex BEFORE word count
        for p,c in self.CP:
            if re.search(p,ql,re.I):
                cost=0.003; self.s["a"]+=1; self.s["ac"]+=cost; return {"d":"api-pro","c":cost}
        # Short non-complex -> local
        if len(ql.split())<=10: self.s["l"]+=1; return {"d":"ollama","c":0}
        # Default medium -> API flash
        cost=0.0003; self.s["a"]+=1; self.s["ac"]+=cost; return {"d":"api-flash","c":cost}

r=HR()
qs=(["Hello!"]*8+["Thanks!"]*5+["What is Python?"]*7+["How much does it cost?"]*6+["Refund policy?"]*4+
    ["Define ML"]*5+["What is RAG?"]*5+["Summarize this article"]*10+["Write Flask hello"]*8+
    ["Explain transformers"]*7+["Compare RAG vs fine-tuning approach for production"]*8+
    ["Design microservice architecture for 10K users"]*7+["Step by step explain attention in detail"]*6+
    ["Write report on AI regulation"]*5+["Analyze code for bugs"]*5+["Debug this error"]*4)
random.shuffle(qs); qs=qs[:100]
for q in qs: r.route(q)
s=r.s; api_all=100*0.001
print("Hybrid Router:")
print(f"  Local: {s['l']}% | API: {s['a']}%")
for n,c in [("All API",api_all),("Hybrid",s["ac"]),("All Local",0)]:
    sv=(1-c/max(api_all,0.0001))*100; m=c/100*10000*30
    print(f"  {n:<20} ${c:.4f}/100req ${m:.2f}/mo {sv:.0f}% saved")
print(f"  Quality: ~{7.5*s['l']/100+9.0*s['a']/100:.1f}/10 weighted avg")


</details>


---
## Exercise 4 (Challenge): Cloud Run Deploy


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class CRO:
    def __init__(self): self.gpu_hr=0.50; self.cold_s=45; self.tps=80
    def monthly(self,daily,ot=200):
        hrs=max(1,daily*ot/(self.tps*3600)); mc=hrs*30*self.gpu_hr+5
        return {"d":daily,"hrs":round(hrs,1),"usd":round(mc,2),"inr":round(mc*84)}
    def sim(self,cold=False,ot=200):
        if cold: lat=self.cold_s*1000+ot/self.tps*1000
        else: lat=ot/self.tps*1000
        return round(max(100,lat+random.gauss(0,50)))

def api_mo(d,ot=200): return round(d*ot*0.25/1e6*30,2)

c=CRO()
print("Ollama on Cloud Run:")
print(f"  Latency: cold={c.sim(True)}ms warm={c.sim(False)}ms")
print(f"\n  {'Daily':>8} {'CR $/mo':>9} {'CR Rs':>8} {'API $':>7} {'API Rs':>8} {'Winner':>7}")
bp=None
for d in [1000,5000,10000,50000,100000]:
    cr=c.monthly(d); api=api_mo(d)
    w="CR" if cr["usd"]<api else "API"
    if w=="CR" and bp is None: bp=d
    print(f"  {d:>8,} ${cr['usd']:>8} Rs{cr['inr']:>6,} ${api:>6} Rs{round(api*84):>7,} {w:>7}")
print(f"\n  Break-even: ~{bp:,}/day" if bp else "\n  Break-even: >100K/day")
print(f"  <10K: API | 10-50K: evaluate | >50K: Cloud Run GPU")


</details>
